Copyright 2025 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left"> <td>      <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Demos/Emoji-Gemma-on-Web/resources/Convert_Gemma_3_270M_to_ONNX.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

# 將 Gemma 3 270M 轉換為 ONNX

此 notebook 將 Gemma 3 模型匯出為 ONNX 格式，以便與 [Transformers.js](https://huggingface.co/docs/transformers.js/en/index) 一起使用，後者使用 ONNX Runtime 在瀏覽器中執行模型。整個過程不到 10 分鐘：
1. 設定Colab環境
2. 從Hugging Face載入模型
3. 使用最佳轉換腳本轉換模型
4. 測試、評估並保存模型以供進一步使用

Gemma 3 270M 專為特定任務fine-tuning 而設計，旨在在行動、網路和邊緣設備上實現高效效能。您可以在此 [notebook](https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Demos/Emoji-Gemma-on-Web/resources/Fine_tune_Gemma_3_270M_for_emoji_generation.ipynb) 中微調您自己的模型，並在轉換後在演示 [網頁應用程式](https://github.com/google-gemini/gemma-cookbook/tree/main/Demos/Emoji-Gemma-on-Web/app-transformersjs) 中執行它。
## 設定開發環境

第一步是使用 pip 安裝軟體包。

In [ ]:
%pip install transformers==4.56.1 onnx==1.19.0 onnx_ir==0.1.7 onnxruntime==1.22.1 numpy==2.3.2 huggingface_hub

重新啟動會話 runtime 以確保您正在使用新安裝的軟體包。

## 轉換模型
若要存取模型並將其儲存至Hugging Face，請提供您的[存取權杖](https://huggingface.co/settings/tokens)。您可以將其儲存為Colab secret 在左側工具列中，方法是將`HF_TOKEN` 指定為「名稱」並新增您唯一的token 作為「值」。

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

您將透過 [Xenova](https://huggingface.co/Xenova) 執行 build_gemma.py 腳本來轉換和量化 Gemma 3 模型。
指定要轉換的模型的 Hugging Face 儲存庫 ID。 .onnx 匯出將儲存到您的Colab 檔案。

In [ ]:
!wget https://gist.githubusercontent.com/xenova/a219dbf3c7da7edd5dbb05f92410d7bd/raw/45f4c5a5227c1123efebe1e36d060672ee685a8e/build_gemma.py

model_author = ""                                         #@param {type:"string"}
gemma_model = "myemoji-gemma-3-270m-it"                   #@param {type:"string"}

repo_id   = f"{model_author}/{gemma_model}"               # Model to convert
save_path = f"/content/{gemma_model}-onnx"                # Path to save resized model

!python build_gemma.py \
    --model_name {repo_id} \
    --output {save_path} \
    -p fp32 fp16 q4 q4f16

print(f"Converted ONNX models saved to {save_path}")

## 測試轉換後的模型

將 .onnx 模型儲存到 Colab 會話後，請嘗試使用 ONNX Runtime 測試 inference。請注意，這可能與用於瀏覽器inference 的ONNX Runtime Web 版本不同。
在 `text_to_translate` 中試驗不同的文字輸入，並探索不同量化版本的表現。

In [ ]:
from transformers import AutoConfig, AutoTokenizer, GenerationConfig
import onnxruntime
import numpy as np

# Load config, processor, and model
config = AutoConfig.from_pretrained(save_path)
generation_config = GenerationConfig.from_pretrained(save_path)
tokenizer = AutoTokenizer.from_pretrained(save_path)

model_file = "onnx/model.onnx"          #@param ["onnx/model.onnx", "onnx/model_fp16.onnx", "onnx/model_q4.onnx", "onnx/model_q4f16.onnx"]

model_path = f"{save_path}/{model_file}"
decoder_session = onnxruntime.InferenceSession(model_path)

## Set config values
num_key_value_heads = config.num_key_value_heads
head_dim = config.head_dim
num_hidden_layers = config.num_hidden_layers
eos_token_id = tokenizer.eos_token_id

# Prepare inputs
text_to_translate = "i love sushi"      # @param {type:"string"}
messages = [
  { "role": "system", "content": "Translate this text to emoji: " },
  { "role": "user", "content": text_to_translate },
]

inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="np")
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']
batch_size = input_ids.shape[0]
past_key_values = {
    f'past_key_values.{layer}.{kv}': np.zeros([batch_size, num_key_value_heads, 0, head_dim], dtype=np.float32)
    for layer in range(num_hidden_layers)
    for kv in ('key', 'value')
}
position_ids = np.tile(np.arange(0, input_ids.shape[-1]), (batch_size, 1))

# 3. Generation loop
max_new_tokens = 8
generated_tokens = np.array([[]], dtype=np.int64)

for i in range(max_new_tokens):
  logits, *present_key_values = decoder_session.run(None, dict(
      input_ids=input_ids,
      attention_mask=attention_mask,
      position_ids=position_ids,
      **past_key_values,
  ))

  ## Update values for next generation loop
  input_ids = logits[:, -1].argmax(-1, keepdims=True)
  attention_mask = np.concatenate([attention_mask, np.ones_like(input_ids, dtype=np.int64)], axis=-1)
  position_ids = position_ids[:, -1:] + 1

  for j, key in enumerate(past_key_values):
    past_key_values[key] = present_key_values[j]

  generated_tokens = np.concatenate([generated_tokens, input_ids], axis=-1)

  if np.isin(input_ids, eos_token_id).any():
    break

# 4. Output result
print(tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0])

## 上傳至Hugging Face Hub

將匯出的 ONNX 模型上傳至Hugging Face，以便於共用和使用。

In [ ]:
import huggingface_hub
from huggingface_hub import whoami

username = whoami()['name']

#@markdown Name your ONNX model:
onnx_model = "myemoji-gemma-3-270m-it-onnx"       #@param {type:"string"}
hf_repo_id = f"{username}/{onnx_model}"

huggingface_hub.create_repo(hf_repo_id, exist_ok=True)

repo_url = huggingface_hub.upload_folder(
  folder_path=save_path,
  repo_id=hf_repo_id,
  repo_type="model",
  commit_message=f"Upload ONNX model files for {onnx_model}"
  )

print(f"Uploaded to {repo_url}")

## 使用 Transformers.js 將模型部署到網絡

現在，您可以使用 [Transformers.js](https://huggingface.co/docs/transformers.js/en/index) 透過 ONNX Runtime Web 在瀏覽器中執行 Gemma 3 模型。 Try it in the [emoji generation web app](https://github.com/google-gemini/gemma-cookbook/tree/main/Demos/Emoji-Gemma-on-Web/app-transformersjs) which runs the model directly in the browser.